In [ ]:
# Imports

import numpy as np
import sympy as sp
from sympy import symbols, Function, diff, tanh, sinh, exp, sqrt, simplify
from sympy.utilities.lambdify import lambdify
from scipy.integrate import solve_ivp


In [ ]:
t = sp.symbols('t')
# Number of params to identify
n_vars = 8
kk = sp.symbols('k1:%d' % (n_vars+1))  # k1, k2, ..., k8

# Constants
Crate = -1

Numexp = 63
Totexp = 3100

N, M, NM = 2, 2, 2

# Design parameters
ep, es, en = 0.335, 0.47, 0.25
brugp, brugs, brugn = 2.43, 2.57, 2.91
lp, ls, ln1 = 75.6e-6, 12e-6, 85.2e-6
Rpp, Rpn = 5.22e-6, 5.86e-6
F = 96487
R_const = 8.3143
t1 = 0.363
ap = (3/Rpp)*(1-ep)
an = (3/Rpn)*(1-en)
T = 298.15
Acell = 0.11
Capa = 5
iapp = Capa * Crate / Acell

# Transport params symbolic with kk
c0 = 1000
D1 = kk[0] * 1e-9
Kappa = kk[1]
ctp = 51765
ctn = 29583
Dbulk = D1
sigmap = kk[2]
sigman = kk[3]
Dsp = kk[4] * 1e-15
Dsn = kk[5] * 1e-14

Keffp = Kappa * (ep ** brugp)
Keffs = Kappa * (es ** brugs)
Keffn = Kappa * (en ** brugn)
D2pos = (ep ** brugp) * Dbulk
D2sep = (es ** brugs) * Dbulk
D2neg = (en ** brugn) * Dbulk

kp = kk[6] * 1e-11
kn = kk[7] * 1e-12

h = lp/(N+1)
h2 = ls/(M+1)
h3 = ln1/(NM+1)


In [ ]:
Nt = 1 + N + 1 + M + 1 + NM + 1 + N + NM + N + NM + N + 2 + NM + 2 + 1 + N + 1 + M + 1 + NM + 1

X = [Function(f'X_{i+1}')(t) for i in range(Nt)]


In [ ]:
# Electrolyte concentration u1 (length = 1+N+1+M+1+NM+1)
u1_len = 1+N+1+M+1+NM+1
u1 = X[0:u1_len]

# Surface concentration u2 (length = N + NM)
u2 = [None] * (N + NM + 1)  # 1-based indexing adjustment, keep element 0 unused or None

for i in range(1, N+1):
    u2[i] = X[i + (N+1) + (M+1) + (NM+1) - 1]  # MATLAB 1-based to Python 0-based adjustment

for i in range(1, NM+1):
    u2[i+N] = X[i + (N+1) + (M+1) + (NM+1) + N - 1]

# Average concentration u3 (length = N + NM)
u3 = [None] * (N + NM + 1)
for i in range(1, N+1):
    u3[i] = X[i + (N+1) + (M+1) + (NM+1) + N + NM - 1]

for i in range(1, NM+1):
    u3[i+N] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N - 1]

# Solid phase potential u4 (length = N + 2 + NM + 2)
u4 = [None] * (N + 2 + NM + 2)
for i in range(1, N+3):
    u4[i-1] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N + NM - 1]

for i in range(1, NM+3):
    u4[i+N+1] = X[i + (N+1) + (M+1) + (NM+1) + N + NM + N + NM + N + 2 - 1]

# Liquid potential u5 (length = 1 + N + 1 + M + 1 + NM + 1)
u5 = [None] * u1_len
offset = (N+1) + (M+1) + (NM+1) + N + NM + N + NM + N + 2 + NM + 2
for i in range(u1_len):
    u5[i] = X[i + offset]


In [ ]:
jp = [None] * (N+2)  # 1-based, jp[0] unused

for i in range(1, N+2):
    theta = u2[i]  # theta = u2(i)*ctp/ctp = u2(i)
    Up = (-0.8090)*theta + 4.4875 - 0.0428*tanh(18.5138*(theta-0.5542)) - 17.7326*tanh(15.7890*(theta-0.3117)) + 17.5842*tanh(15.9308*(theta-0.3120))
    jp[i] = 2*kp*sqrt(u1[i]*c0)*sqrt(ctp - u2[i]*ctp)*sqrt(u2[i]*ctp)*sinh(0.5*F/(R_const*T)*(u4[i] - u5[i] - Up))
